<a href="https://colab.research.google.com/github/ElionLAB/OOP_2026_Practice/blob/main/ch_07/src/part_1/student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Environment Setup — Auto-detect Google Colab / Local
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    pass
else:
    print('Local environment: Make sure `conda activate oop_practice` is active.')

# Lecture 7 — Part 1 (Slides 1–44): The Iterator Pattern, Comprehensions, Generators, Pipelines & K-NN

**Konkuk University OOP (Python Object-Oriented Programming) — Spring 2026**

---

## Learning Objectives

1. Implement the Iterator Protocol from scratch (`__iter__`, `__next__`, `StopIteration`) and drive it with a manual `for`-loop.
2. Refactor procedural append loops into **list comprehensions** — basic and filtered.
3. Use **set** and **dict** comprehensions with `namedtuple` records.
4. Switch from eager `[...]` to lazy `(...)` **generator expressions** for huge data.
5. Use the **`yield`** keyword to build stateful generators that process streams.
6. Use **`yield from`** to delegate to sub-iterators.
7. Compose **generator pipelines** — chained generators that consume a 1TB file with O(1) RAM.
8. Build the K-NN classifier: `defaultdict` bucketing → `itertools.chain` split → brute-force `k_nn_standard` → optimized `k_nn_bisect` → `Hyperparameter.test()` accuracy → `timeit` comparison.

## How this notebook is structured

Each section follows the same rhythm:

1. **Concept** — short explanation from the slide.
2. **Full code reference** — the *goal* code block.
3. **TODO** cell — fill the small blanks marked `# TODO`.
4. **Quick check** — assertions verify your solution.

> **Continuity with Lectures 5, 6.1, and 6.2.** Lecture 5 §7 built the K-NN train/test split as a *class*. 6.1 §8 functionalized it into a higher-order `partition`. 6.2 §9 provided the ingestion readers. **This notebook §8 closes the loop**: it actually *uses* that split for classification, with two competing algorithms compared by `timeit`.

## 1. The Iterator Protocol (Slides 3–8)

> *"An iterator is an object that provides two methods: `next()` to retrieve the next item, `done()` to check if the sequence is exhausted. Python simplifies this — the `for` loop manages iteration automatically, hiding the `while` boilerplate."* — Slide 3

The Python protocol (slide 5):

| Step | Method                                | Where it lives        |
| ---- | ------------------------------------- | --------------------- |
| 1    | `__iter__()` returns the iterator     | on the **Iterable**   |
| 2    | `__next__()` returns the next element | on the **Iterator**   |
| 3    | raise `StopIteration` when exhausted  | inside `__next__`     |

### 1.1 — Building `CapitalIterable` + `CapitalIterator` (Slide 6)

Two-class pattern: the **Iterable** is the "collection" object (knows its data), and `__iter__()` constructs a fresh **Iterator** every time. The Iterator carries the *cursor* (`self.index`) and raises `StopIteration` when it runs out.

### Full code reference (Slide 6)

```python
class CapitalIterable:
    def __init__(self, string: str):
        self.string = string

    def __iter__(self):
        return CapitalIterator(self.string)


class CapitalIterator:
    def __init__(self, string: str):
        self.words = [w.capitalize() for w in string.split()]
        self.index = 0

    def __next__(self):
        if self.index == len(self.words):
            raise StopIteration()
        word = self.words[self.index]
        self.index += 1
        return word
```

In [ ]:
# TODO 1.1 — Implement the Iterable + Iterator pair

class CapitalIterable:
    def __init__(self, string: str):
        # TODO 1.1-a: store the raw string on self
        self.string = ...

    def __iter__(self):
        # TODO 1.1-b: return a NEW CapitalIterator built from self.string
        return ...


class CapitalIterator:
    def __init__(self, string: str):
        # TODO 1.1-c: split the string into words and capitalize each — store as self.words
        # Hint: list comprehension over string.split()
        self.words = ...
        # TODO 1.1-d: start the cursor self.index at 0
        self.index = ...

    def __next__(self):
        # TODO 1.1-e: when self.index reaches len(self.words), raise StopIteration()
        if ...:
            raise StopIteration()
        # TODO 1.1-f: grab the word at self.index, advance index by 1, and return the word
        word = ...
        self.index += 1
        return word

In [ ]:
# Quick check — drive it with a normal for-loop (Python wires iter()/next() implicitly).
result = list(CapitalIterable('the quick brown fox'))
assert result == ['The', 'Quick', 'Brown', 'Fox'], result
print(f"OK: list(CapitalIterable(...)) → {result}")

# A second pass must work — Iterables can be looped multiple times (slide 7).
second_pass = list(CapitalIterable('hello world'))
assert second_pass == ['Hello', 'World']
print(f"OK: a fresh pass also works → {second_pass}")
print("Section 1.1 passed.")

### 1.2 — Under the Hood of a `for` Loop (Slide 8)

What `for x in iterable:` *really* does:

1. Call `iter(iterable)` → bind the iterator.
2. Loop: call `next(iterator)`.
3. On `StopIteration` → exit.

Doing this by hand makes the protocol visible.

### Full code reference (Slide 8)

```python
iterable = CapitalIterable('the quick brown fox')
iterator = iter(iterable)   # Calls __iter__()
while True:
    try:
        print(next(iterator))   # Calls __next__()
    except StopIteration:
        break
# Output: The / Quick / Brown / Fox
```

In [ ]:
# TODO 1.2 — Drive the iterator manually (no for-loop)

iterable = CapitalIterable('the quick brown fox')

# TODO 1.2-a: call iter() on the iterable to get an iterator
iterator = ...

collected = []
# Safety guard: slide 8 uses `while True:`. We use a bounded for-loop here so
# the cell cannot hang if you Run All before filling the TODOs in.
for _ in range(50):
    try:
        # TODO 1.2-b: call next(iterator) and bind it to `word`
        word = ...
    # TODO 1.2-c: catch StopIteration so the loop exits cleanly
    except ...:
        break
    collected.append(word)
    print(word)
else:
    raise RuntimeError("Loop ran 50 times without StopIteration — did you complete the TODOs?")


In [ ]:
# Quick check
assert collected == ['The', 'Quick', 'Brown', 'Fox'], collected
print(f"OK: hand-driven loop collected {collected}")
print("Section 1.2 passed.")

## 2. List Comprehensions (Slides 9–13)

> *"Comprehensions are concise syntax shortcuts for creating a new collection by mapping and filtering an existing iterable. They use the exact same Iterator Protocol as a for-loop, but execute at C-speed."* — Slide 9

Anatomy (slide 12):

```
[ expression  for item in iterable  if condition ]
   └─output─┘ └────loop─────────┘ └─filter─┘
```

### 2.1 — Refactoring a Procedural Loop (Slides 10–11)

Replace the `[]` + `for` + `append()` trio with a single bracket expression.

### Full code reference (Slide 11)

```python
input_strings = ['1', '5', '28', '131', '3']

# The verbose procedural way
output_integers = []
for num in input_strings:
    output_integers.append(int(num))

# The List Comprehension way
output_integers = [int(num) for num in input_strings]
print(output_integers)   # [1, 5, 28, 131, 3]
```

In [ ]:
# TODO 2.1 — Convert strings to integers with a list comprehension

input_strings = ['1', '5', '28', '131', '3']

# TODO 2.1-a: build output_integers in ONE line using a list comprehension
output_integers = ...

print(output_integers)

In [ ]:
# Quick check
assert output_integers == [1, 5, 28, 131, 3], output_integers
assert all(isinstance(x, int) for x in output_integers)
print(f"OK: {output_integers}")
print("Section 2.1 passed.")

### 2.2 — Filtering + TSV Parsing (Slide 13)

Two more advanced patterns: a `range` + `if` filter, and a `dict(zip(...))` per line for tabular data.

### Full code reference (Slide 13)

```python
# Filtering only even numbers during list creation
even_squares = [x**2 for x in range(10) if x % 2 == 0]
print(even_squares)   # [0, 4, 16, 36, 64]

# Processing a tab-separated file line by line
with open("data.tsv") as file:
    header = file.readline().strip().split('\t')
    contacts = [
        dict(zip(header, line.strip().split('\t')))
        for line in file
    ]
```

In [ ]:
# Setup — write a tiny tab-separated file we can really parse
from pathlib import Path
Path("data.tsv").write_text(
    "name\temail\tage\n"
    "Alice\talice@example.com\t30\n"
    "Bob\tbob@example.com\t25\n"
    "Carol\tcarol@example.com\t40\n",
    encoding="utf-8",
)
print("Wrote data.tsv (1 header + 3 rows)")

In [ ]:
# TODO 2.2 — Filter comprehension + TSV parsing

# TODO 2.2-a: build a list of x squared for x in 0..9, KEEPING only even x
even_squares = ...

print(even_squares)

with open("data.tsv") as file:
    # TODO 2.2-b: read the first line, strip the trailing newline, split on '\t'
    header = ...

    # TODO 2.2-c: list comprehension — for each remaining line, build dict(zip(header, fields))
    #             where fields = line.strip().split('\t')
    contacts = ...

for c in contacts:
    print(c)

In [ ]:
# Quick check
assert even_squares == [0, 4, 16, 36, 64], even_squares
assert header == ['name', 'email', 'age'], header
assert contacts == [
    {'name': 'Alice', 'email': 'alice@example.com', 'age': '30'},
    {'name': 'Bob',   'email': 'bob@example.com',   'age': '25'},
    {'name': 'Carol', 'email': 'carol@example.com', 'age': '40'},
], contacts
print(f"OK: even_squares = {even_squares}")
print(f"OK: header       = {header}")
print(f"OK: parsed {len(contacts)} contacts as dicts")
print("Section 2.2 passed.")

## 3. Set & Dict Comprehensions (Slides 14–15)

| Syntax                            | Builds       |
| --------------------------------- | ------------ |
| `[expr for x in it]`              | **list**     |
| `(expr for x in it)`              | **generator**|
| `{expr for x in it}`              | **set**      |
| `{k: v for x in it}`              | **dict**     |

Sets automatically **deduplicate**. Dicts need the `key: value` colon.

### Full code reference (Slide 15)

```python
from collections import namedtuple

Book = namedtuple("Book", "author title genre")
books = [
    Book("Pratchett", "Nightwatch",       "fantasy"),
    Book("Pratchett", "Thief Of Time",    "fantasy"),
    Book("Le Guin",   "The Dispossessed", "scifi"),
]

# Set Comprehension: Extracts unique authors
fantasy_authors = {b.author for b in books if b.genre == 'fantasy'}
# {'Pratchett'}

# Dictionary Comprehension: Maps title to book object
titles_map = {b.title: b for b in books if b.genre == 'fantasy'}
```

In [ ]:
# Setup — Book records exactly as the slide
from collections import namedtuple

Book = namedtuple("Book", "author title genre")
books = [
    Book("Pratchett", "Nightwatch",       "fantasy"),
    Book("Pratchett", "Thief Of Time",    "fantasy"),
    Book("Le Guin",   "The Dispossessed", "scifi"),
]
print(books)

In [ ]:
# TODO 3.1 — Set comprehension: unique fantasy authors

# TODO 3.1-a: build a SET of b.author for each book with genre == 'fantasy'
fantasy_authors = ...

print(fantasy_authors)

In [ ]:
# Quick check
assert isinstance(fantasy_authors, set), f"expected set, got {type(fantasy_authors).__name__}"
assert fantasy_authors == {'Pratchett'}, fantasy_authors
print(f"OK: fantasy_authors = {fantasy_authors}  (dedup worked — Pratchett only listed once)")
print("Section 3.1 passed.")

In [ ]:
# TODO 3.2 — Dict comprehension: map title → Book

# TODO 3.2-a: build a DICT mapping b.title → b for each book with genre == 'fantasy'
titles_map = ...

for title, book in titles_map.items():
    print(f"  {title!r} → {book}")

In [ ]:
# Quick check
assert isinstance(titles_map, dict)
assert set(titles_map.keys()) == {"Nightwatch", "Thief Of Time"}
assert titles_map["Nightwatch"].author == "Pratchett"
assert titles_map["Thief Of Time"].genre == "fantasy"
print(f"OK: titles_map has {len(titles_map)} fantasy titles, each mapped to its Book record")
print("Section 3.2 passed.")

## 4. Generator Expressions — Lazy Evaluation (Slides 16–17)

| Syntax     | Evaluation | Memory                     | Best for                  |
| ---------- | ---------- | -------------------------- | ------------------------- |
| `[expr …]` | eager      | builds full list (≈ N × item) | small data, indexing      |
| `(expr …)` | lazy       | yields one at a time (≈ const) | huge files, streams       |

### 4.1 — Memory Comparison (Slide 16) — Observation Cell

The list comprehension allocates the entire collection in RAM up front. The generator expression allocates a single tiny iterator object (~112 bytes regardless of N).

In [ ]:
# Observation only — no TODO. Run it and compare the two sizes.
import sys

big_list = [x for x in range(1_000_000)]
big_gen  = (x for x in range(1_000_000))

list_bytes = sys.getsizeof(big_list)
gen_bytes  = sys.getsizeof(big_gen)

print(f"List comprehension over 1M items: {list_bytes:>10,} bytes")
print(f"Generator expression same N    : {gen_bytes:>10,} bytes")
print(f"Ratio (list / gen)             : {list_bytes / gen_bytes:>10,.0f}×")
# Take-away: the generator is ~constant size; the list grows linearly with N.

### 4.2 — Lazy File Filter (Slide 17)

A generator expression over a file object means we hold **one line** at a time, even if the file is a terabyte.

### Full code reference (Slide 17)

```python
import sys
inname = "massive_server_log.txt"
outname = "warnings_only.log"

with open(inname) as infile:
    with open(outname, "w") as outfile:
        # Generator Expression: Evaluates lazily
        warnings = (l for l in infile if 'WARNING' in l)
        # Pulls data one line at a time
        for l in warnings:
            outfile.write(l)
```

In [ ]:
# Setup — write a tiny log file that we can pretend is "massive"
from pathlib import Path

Path("massive_server_log.txt").write_text(
    "INFO    server started\n"
    "WARNING disk usage at 80%\n"
    "INFO    user logged in\n"
    "WARNING memory pressure rising\n"
    "ERROR   uncaught exception in worker\n"
    "WARNING cache miss spike\n"
    "INFO    nightly job complete\n",
    encoding="utf-8",
)
print("Wrote massive_server_log.txt (7 lines)")

In [ ]:
# TODO 4.2 — Lazy filter of warnings into a separate file

inname = "massive_server_log.txt"
outname = "warnings_only.log"

with open(inname) as infile:
    with open(outname, "w") as outfile:
        # TODO 4.2-a: generator expression — keep only lines that contain 'WARNING'
        warnings = ...

        # TODO 4.2-b: drain the generator and write each kept line to outfile
        for l in warnings:
            ...

In [ ]:
# Quick check
written = Path(outname).read_text(encoding="utf-8").splitlines()
assert written == [
    "WARNING disk usage at 80%",
    "WARNING memory pressure rising",
    "WARNING cache miss spike",
], written
print(f"OK: lazy filter kept {len(written)} WARNING lines:")
for w in written:
    print("   ", w)

# Confirm the type: a generator expression is a `generator`, NOT a list.
import types
with open(inname) as f:
    g = (l for l in f if 'WARNING' in l)
    assert isinstance(g, types.GeneratorType), f"got {type(g)}"
print("OK: the filter object is a real <class 'generator'>, not a list")
print("Section 4.2 passed.")

## 5. `yield` — Generator Functions (Slides 18–21)

> **Factory worker analogy** (slide 18): instead of building 100 toys and handing you a box, the worker builds ONE toy, hands it to you, and hits *pause*. When you ask for another, they resume from exactly where they left off.

The `yield` keyword transforms a function into a stateful **generator object**. Calling the function does NOT execute the body — it returns a paused generator.

### 5.1 — Defining `warnings_filter` (Slide 19)

`yield` here works almost like the gen-expression in §4, but inside a regular `def` — which means you can interleave arbitrary Python logic between yields (logging, formatting, exception handling, etc.).

### Full code reference (Slide 19)

```python
def warnings_filter(insequence):
    for line in insequence:
        if 'WARNING' in line:
            # Pauses execution and returns
            yield line.replace('\tWARNING', '')

# Creating the generator object (Code does NOT run yet)
filter_gen = warnings_filter(open("log.txt"))
print(type(filter_gen))
# <class 'generator'>
```

In [ ]:
# TODO 5.1 — Define warnings_filter as a generator function

def warnings_filter(insequence):
    # TODO 5.1-a: iterate over each line in insequence
    for line in ...:
        # TODO 5.1-b: keep ONLY lines containing 'WARNING'
        if ...:
            # TODO 5.1-c: yield the line with '\tWARNING' stripped out
            yield ...

In [ ]:
# Quick check — calling the function should NOT execute the body.
# It just constructs a paused generator object.
gen = warnings_filter(iter(["INFO foo", "\tWARNING disk", "\tWARNING ram", "ERROR bar"]))
import types
assert isinstance(gen, types.GeneratorType), f"got {type(gen).__name__}"
print(f"OK: warnings_filter(...) returned a {type(gen).__name__}, not a list")

# Now drain it
out = list(gen)
assert out == [' disk', ' ram'], out
print(f"OK: drained generator → {out}")
print("Section 5.1 passed.")

### 5.2 — Consuming a Generator Across Two Files (Slide 21)

The generator is wired between an input file and an output file. Memory stays constant — we never materialize the full log.

### Full code reference (Slide 21)

```python
def warnings_filter(insequence):
    for line in insequence:
        if 'WARNING' in line:
            yield line.replace('\tWARNING', '')


with open("input.txt") as infile:
    with open("output.txt", "w") as outfile:
        # Create generator
        filter_gen = warnings_filter(infile)
        # Consume generator incrementally
        for l in filter_gen:
            outfile.write(l)
```

In [ ]:
# Setup — write an input log with literal \tWARNING segments
from pathlib import Path

Path("input.txt").write_text(
    "INFO\tstart\n"
    "INFO\tWARNING disk pressure rising\n"      # has \tWARNING — keep, strip it
    "INFO\tWARNING memory low\n"                 # has \tWARNING — keep, strip it
    "ERROR\tboom\n"
    "INFO\tend\n",
    encoding="utf-8",
)
print("Wrote input.txt")

In [ ]:
# TODO 5.2 — Wire warnings_filter between input.txt and output.txt

with open("input.txt") as infile:
    with open("output.txt", "w") as outfile:
        # TODO 5.2-a: create the generator by calling warnings_filter on infile
        filter_gen = ...

        # TODO 5.2-b: consume the generator and write each yielded line into outfile
        for l in ...:
            outfile.write(l)

In [ ]:
# Quick check
out_lines = Path("output.txt").read_text(encoding="utf-8").splitlines()
# Each kept line had "INFO\tWARNING <msg>"; the generator removed "\tWARNING",
# so the surviving line is "INFO <msg>" (with the LITERAL space before the original word).
assert out_lines == ["INFO disk pressure rising", "INFO memory low"], out_lines
print(f"OK: output.txt contains {len(out_lines)} stripped warning lines:")
for l in out_lines:
    print("   ", l)
print("Section 5.2 passed.")

## 6. `yield from` Delegation (Slides 22–23)

> *"`yield from` provides a syntactic shortcut for yielding all values from an inner iterator directly to the outer caller."* — Slide 22

```python
# Before (Python 3.2)             # After (Python 3.3+)
for item in some_iter:            yield from some_iter
    yield item
```

### Full code reference (Slide 23)

```python
def warnings_filter(infilename):
    with open(infilename) as infile:
        # Delegate yielding to the expression
        yield from (
            line.replace('\tWARNING', '')
            for line in infile
            if 'WARNING' in line
        )


filter_gen = warnings_filter("log.txt")
for l in filter_gen:
    print(l)
```

In [ ]:
# TODO 6.1 — Refactor warnings_filter with `yield from`

def warnings_filter_v2(infilename):
    with open(infilename) as infile:
        # TODO 6.1-a: delegate to a generator EXPRESSION that
        #   - iterates over each line in infile
        #   - keeps only lines containing 'WARNING'
        #   - replaces '\tWARNING' with '' before yielding
        # Use `yield from (...)`
        yield from ...

In [ ]:
# Quick check — same input/output as §5.2, just rewired through yield from
from pathlib import Path

out_lines_v2 = list(warnings_filter_v2("input.txt"))
assert out_lines_v2 == ["INFO disk pressure rising\n", "INFO memory low\n"], out_lines_v2
print(f"OK: yield-from version produced the same 2 lines as the explicit-loop version")
for l in out_lines_v2:
    print("   ", repr(l))
print("Section 6.1 passed.")

## 7. Generator Pipelines (Slides 24–27)

```
Raw File Iter  →  Filter Gen  →  Split Gen  →  Tuple Gen  →  CSV Writer
```

Each stage is one tiny generator. The pipeline holds **one line** in memory at any moment — even on a 1TB file. Stages are independently unit-testable.

### Full code reference (Slides 25 & 27)

```python
# Generator 1: Read lines
lines_iter = open("log.txt")
# Generator 2: Filter warnings
warnings_iter = (l for l in lines_iter if "WARN" in l)
# Generator 3: Split into time, level, message
split_iter = (l.rstrip().split(" ", 2) for l in warnings_iter)
# Generator 4: Keep complete (time, level, message) tuples
tuple_iter = (tuple(parts) for parts in split_iter if len(parts) == 3)

import csv
with open("output.csv", "w") as target:
    writer = csv.writer(target)
    # This single loop drives the entire stack
    for line_tuple in tuple_iter:
        writer.writerow(line_tuple)
```

In [ ]:
# Setup — log.txt with a mix of INFO and WARN lines in "time level message" format
from pathlib import Path

Path("log.txt").write_text(
    "10:00:01 INFO system started\n"
    "10:00:05 WARN disk usage 80%\n"
    "10:00:10 INFO user login\n"
    "10:00:15 WARN memory pressure rising\n"
    "10:00:20 ERROR uncaught exception\n"
    "10:00:25 WARN cache miss spike\n"
    "10:00:30 INFO nightly job complete\n",
    encoding="utf-8",
)
print("Wrote log.txt (7 lines)")

In [ ]:
# TODO 7.1 — Build the 4-stage pipeline (slide 25)

# TODO 7.1-a: Generator 1 — open log.txt (file objects are already iterators of lines)
lines_iter = ...

# TODO 7.1-b: Generator 2 — keep only lines containing "WARN"
warnings_iter = ...

# TODO 7.1-c: Generator 3 — for each line, rstrip then split with maxsplit=2 on " "
split_iter = ...

# TODO 7.1-d: Generator 4 — keep only lists with exactly 3 parts, convert each to a tuple
tuple_iter = ...

In [ ]:
# TODO 7.2 — Drive the pipeline by writing it to output.csv (slide 27)
import csv

with open("output.csv", "w", newline="") as target:
    # TODO 7.2-a: build a csv.writer on target
    writer = ...
    # TODO 7.2-b: loop over tuple_iter and write each tuple as a row
    for line_tuple in ...:
        ...

In [ ]:
# Quick check — drain output.csv and verify the pipeline kept exactly the WARN rows.
import csv

with open("output.csv", newline="") as f:
    rows = list(csv.reader(f))

assert rows == [
    ["10:00:05", "WARN", "disk usage 80%"],
    ["10:00:15", "WARN", "memory pressure rising"],
    ["10:00:25", "WARN", "cache miss spike"],
], rows
print(f"OK: pipeline kept {len(rows)} WARN rows, properly split into (time, level, message):")
for r in rows:
    print("   ", r)
print("Section 7.1 + 7.2 passed.")

## 8. Case Study — K-Nearest Neighbors (Slides 28–39, 41)

### Where this fits in the K-NN tetralogy

| Lecture       | Layer        | What it produced                                              |
| ------------- | ------------ | ------------------------------------------------------------- |
| 5 §7          | Split (OOP)  | `CountingDealingPartition` class                              |
| 6.1 §8        | Split (func) | higher-order `partition(samples, rule)`                       |
| 6.2 §9        | Ingest       | 4 readers + jsonschema validator                              |
| **7.1 §8**    | **Classify** | **bucketing → chain split → brute/bisect K-NN → accuracy**    |

### The K-NN algorithm (slide 28, 36)

1. **Partition** dataset → training + testing iterators.
2. **Deduplicate** identical samples (bucket them — see §8.1).
3. **Distance** measure between sample pairs.
4. **Classify** the unknown by majority vote of its K nearest neighbors.

### 8.0 — Setup: `Sample`, `manhattan_distance`, `Hyperparameter`, mini-iris data

These pieces are provided (not TODOs). The slide deck doesn't show `Sample`, `manhattan_distance`, or the `Hyperparameter` class body — it only shows how they're *used*. We provide minimal implementations matching the slide's usage exactly:

- `Sample` — a `namedtuple` with `sepal_len, petal_len, species`. Slide 31 uses `sample.sepal_len, sample.petal_len`; slide 37 uses `s.species`.
- `manhattan_distance(s1, s2)` — sum of absolute differences across `sepal_len` and `petal_len` (slide 35 just names it).
- `Hyperparameter(k, distance, training, algorithm)` with `.test(testing_data)` returning accuracy in `[0, 1]` (slide 35).
- 30 iris-like samples for the case study (extension of the 10-row mini-iris from Lecture 6.2 — we need duplicates and more data for K-NN to be interesting).

In [ ]:
# Setup — provided, not a TODO. Run as-is.
from collections import namedtuple
import statistics

Sample = namedtuple("Sample", ["sepal_len", "petal_len", "species"])


def manhattan_distance(s1: Sample, s2: Sample) -> float:
    """Sum of absolute differences across the two measurement axes."""
    return abs(s1.sepal_len - s2.sepal_len) + abs(s1.petal_len - s2.petal_len)


class Hyperparameter:
    """Holds K-NN configuration and runs it against a test set (slide 34–35)."""
    def __init__(self, k, distance, training, algorithm):
        self.k = k
        self.distance = distance
        self.training = training
        self.algorithm = algorithm

    def test(self, testing_data) -> float:
        if not testing_data:
            return 0.0
        correct = sum(
            1 for s in testing_data
            if self.algorithm(self.k, self.distance, s, self.training) == s.species
        )
        return correct / len(testing_data)


# 30 iris-like samples — three species, with some intentional duplicates
# (slide 30 demands deduplication; bucketing has to actually collide on something).
ALL_SAMPLES = [
    Sample(5.1, 1.4, "Iris-setosa"),  Sample(4.9, 1.4, "Iris-setosa"),
    Sample(4.7, 1.3, "Iris-setosa"),  Sample(4.6, 1.5, "Iris-setosa"),
    Sample(5.0, 1.4, "Iris-setosa"),  Sample(5.1, 1.4, "Iris-setosa"),  # dup of #0
    Sample(5.4, 1.7, "Iris-setosa"),  Sample(4.8, 1.4, "Iris-setosa"),
    Sample(5.0, 1.4, "Iris-setosa"),  Sample(4.4, 1.3, "Iris-setosa"),  # dup of #4 measurements
    Sample(7.0, 4.7, "Iris-versicolor"), Sample(6.4, 4.5, "Iris-versicolor"),
    Sample(6.9, 4.9, "Iris-versicolor"), Sample(5.5, 4.0, "Iris-versicolor"),
    Sample(6.5, 4.6, "Iris-versicolor"), Sample(5.7, 4.5, "Iris-versicolor"),
    Sample(6.3, 4.7, "Iris-versicolor"), Sample(4.9, 3.3, "Iris-versicolor"),
    Sample(6.6, 4.6, "Iris-versicolor"), Sample(5.2, 3.9, "Iris-versicolor"),
    Sample(6.3, 6.0, "Iris-virginica"), Sample(5.8, 5.1, "Iris-virginica"),
    Sample(7.1, 5.9, "Iris-virginica"), Sample(6.3, 5.6, "Iris-virginica"),
    Sample(6.5, 5.8, "Iris-virginica"), Sample(7.6, 6.6, "Iris-virginica"),
    Sample(4.9, 4.5, "Iris-virginica"), Sample(7.3, 6.3, "Iris-virginica"),
    Sample(6.7, 5.8, "Iris-virginica"), Sample(7.2, 6.1, "Iris-virginica"),
]
print(f"Loaded {len(ALL_SAMPLES)} samples across "
      f"{len({s.species for s in ALL_SAMPLES})} species.")

### 8.1 — Bucketing with `defaultdict` (Slides 30–31)

> Slide 30: *"We cannot simply slice the list, as duplicate samples might end up in BOTH training and testing sets, ruining the evaluation."*

Solution: hash each sample's measurements, mod by 60 → bucket id. Samples with identical measurements ALWAYS land in the same bucket — no leakage.

### Full code reference (Slide 31)

```python
from collections import defaultdict
from typing import DefaultDict, List

# Type alias for clarity
ModuloDict = DefaultDict[int, List[Sample]]

def partition_data(samples):
    # Group samples by measurements
    buckets: ModuloDict = defaultdict(list)
    for sample in samples:
        key = hash((sample.sepal_len, sample.petal_len))
        buckets[key % 60].append(sample)
    return buckets
```

In [ ]:
# TODO 8.1 — Bucketing partition
from collections import defaultdict


def partition_data(samples):
    # TODO 8.1-a: create a defaultdict whose default factory is `list`
    buckets = ...

    for sample in samples:
        # TODO 8.1-b: compute a hash of (sample.sepal_len, sample.petal_len)
        key = ...
        # TODO 8.1-c: append sample to the bucket at index key % 60
        ...

    return buckets

In [ ]:
# Quick check
buckets = partition_data(ALL_SAMPLES)
total_in_buckets = sum(len(v) for v in buckets.values())
assert total_in_buckets == len(ALL_SAMPLES), \
    f"every sample must land somewhere: {total_in_buckets} vs {len(ALL_SAMPLES)}"
print(f"OK: {len(ALL_SAMPLES)} samples spread across {len(buckets)} buckets")

# The key property: identical measurements share a bucket.
# Sample(5.1, 1.4, ...) appears twice → they MUST be in the same bucket.
same_meas_key = hash((5.1, 1.4)) % 60
same_meas_bucket = buckets[same_meas_key]
same_meas_count = sum(1 for s in same_meas_bucket if (s.sepal_len, s.petal_len) == (5.1, 1.4))
assert same_meas_count == 2, f"both copies of (5.1, 1.4) should share bucket {same_meas_key}, got {same_meas_count}"
print(f"OK: both samples with measurements (5.1, 1.4) landed in the SAME bucket — no leakage possible")
print("Section 8.1 passed.")

### 8.2 — Flattening with `itertools.chain` (Slides 32–33)

> Slide 32: `chain` consumes multiple iterators and yields their items as one continuous sequence.

Strategy: bucket ids `0, 5, 10, …` → testing (1-in-5 split). Everything else → training. Then `chain(*buckets)` flattens the bucket-of-buckets back into a single stream of samples.

### Full code reference (Slide 33)

```python
import itertools

def split_partitions(buckets):
    # Generator expressions grabbing sub-lists
    test_buckets  = (buckets[i] for i in range(60) if i % 5 == 0)
    train_buckets = (buckets[i] for i in range(60) if i % 5 != 0)
    # Chain links into a single flat stream
    testing_data  = list(itertools.chain(*test_buckets))
    training_data = list(itertools.chain(*train_buckets))
    return testing_data, training_data
```

In [ ]:
# TODO 8.2 — Split buckets into training and testing via itertools.chain
import itertools


def split_partitions(buckets):
    # TODO 8.2-a: generator expression — yield buckets[i] for i in 0..59 WHERE i % 5 == 0
    test_buckets  = ...
    # TODO 8.2-b: generator expression — yield buckets[i] for i in 0..59 WHERE i % 5 != 0
    train_buckets = ...
    # TODO 8.2-c: flatten test_buckets through itertools.chain into a list
    testing_data  = ...
    # TODO 8.2-d: flatten train_buckets through itertools.chain into a list
    training_data = ...
    return testing_data, training_data

In [ ]:
# Quick check
testing_data, training_data = split_partitions(buckets)
# Every original sample must be in exactly one of the two sets.
assert len(testing_data) + len(training_data) == len(ALL_SAMPLES)
print(f"OK: {len(training_data)} train / {len(testing_data)} test  (total {len(ALL_SAMPLES)})")

# No leakage: a (sepal_len, petal_len) pair in test must NOT appear in train.
test_meas  = {(s.sepal_len, s.petal_len) for s in testing_data}
train_meas = {(s.sepal_len, s.petal_len) for s in training_data}
overlap = test_meas & train_meas
assert overlap == set(), f"LEAKAGE: measurements appearing in both sides: {overlap}"
print(f"OK: no overlap between train/test measurements — slide-30 invariant holds")
print("Section 8.2 passed.")

### 8.3 — Brute-Force `k_nn_standard` (Slides 36–37)

For each test point: compute distance to *every* training point, sort the whole list, take the top K, return the most common species.

### Full code reference (Slide 37)

```python
def k_nn_standard(k, dist_func, unknown, training_data):
    # Generator expression for all distances
    distances = (
        (dist_func(unknown, known), known)
        for known in training_data
    )
    # Sorts the ENTIRE dataset in memory
    sorted_distances = sorted(distances, key=lambda x: x)
    # Extract the k-nearest
    k_nearest = sorted_distances[:k]
    # Return the most common species (Mode)
    return mode([s.species for d, s in k_nearest])
```

In [ ]:
# TODO 8.3 — Brute-force K-NN
from statistics import mode


def k_nn_standard(k, dist_func, unknown, training_data):
    # TODO 8.3-a: generator expression — yield (dist_func(unknown, known), known) for each known in training_data
    distances = ...

    # TODO 8.3-b: sort the distances (key=lambda x: x — slide 37 uses identity ordering)
    sorted_distances = ...

    # TODO 8.3-c: slice the first k items
    k_nearest = ...

    # TODO 8.3-d: return mode(...) over the species of those k items
    #             k_nearest is a list of (distance, sample) tuples — pull s.species from each.
    return ...

In [ ]:
# Quick check — known sample should resolve to itself.
unknown = Sample(5.1, 1.4, "?")   # matches an actual training sample exactly
predicted = k_nn_standard(k=5, dist_func=manhattan_distance,
                          unknown=unknown, training_data=training_data)
assert predicted == "Iris-setosa", predicted
print(f"OK: k_nn_standard predicted {predicted!r} for measurement (5.1, 1.4)")

# Try a clearly-virginica point
unknown_v = Sample(7.2, 6.0, "?")
predicted_v = k_nn_standard(k=5, dist_func=manhattan_distance,
                            unknown=unknown_v, training_data=training_data)
assert predicted_v == "Iris-virginica", predicted_v
print(f"OK: k_nn_standard predicted {predicted_v!r} for measurement (7.2, 6.0)")
print("Section 8.3 passed.")

### 8.4 — Optimized `k_nn_bisect` (Slides 38–39)

| Variant          | Memory     | Time       |
| ---------------- | ---------- | ---------- |
| `sorted()`       | **O(N)**   | O(N log N) |
| `bisect.insort()`| **O(K)**   | O(N × K)   |

The bisect version maintains a small sorted list of size K. When a new candidate comes in: binary-insert, then pop the worst.

### Full code reference (Slide 39)

```python
import bisect

def k_nn_bisect(k, dist_func, unknown, training_data):
    k_nearest = []
    for known in training_data:
        distance = dist_func(unknown, known)
        # Binary insertion keeps the list sorted
        bisect.insort(k_nearest, (distance, known))
        # Trim to only keep Top K elements
        if len(k_nearest) > k:
            k_nearest.pop()
    return mode([s.species for d, s in k_nearest])
```

In [ ]:
# TODO 8.4 — Bisect-optimized K-NN
import bisect


def k_nn_bisect(k, dist_func, unknown, training_data):
    k_nearest = []
    for known in training_data:
        # TODO 8.4-a: compute the distance between unknown and known
        distance = ...

        # TODO 8.4-b: binary-insert (distance, known) into k_nearest so it stays sorted
        ...

        # TODO 8.4-c: if k_nearest has more than k elements, pop the LAST (worst) one
        if ...:
            ...

    # TODO 8.4-d: return mode over the species of the k retained samples
    return ...

In [ ]:
# Quick check — must produce IDENTICAL predictions to the brute-force version.
for unknown in testing_data:
    std = k_nn_standard(k=5, dist_func=manhattan_distance,
                        unknown=unknown, training_data=training_data)
    bis = k_nn_bisect (k=5, dist_func=manhattan_distance,
                        unknown=unknown, training_data=training_data)
    assert std == bis, f"disagreement on {unknown}: standard={std!r}, bisect={bis!r}"
print(f"OK: standard and bisect produced identical predictions on all {len(testing_data)} test samples")
print("Section 8.4 passed.")

### 8.5 — Driving the K-NN Test with `Hyperparameter` (Slides 34–35)

`Hyperparameter` bundles the four knobs of the test run: K, distance function, training set, and algorithm. The slide-35 call shows exactly which keyword args to use.

### Full code reference (Slide 35)

```python
# Create a hyperparameter configuration object
h = Hyperparameter(
    k=5,
    distance=manhattan_distance,
    training=training_data,
    algorithm=k_nn_standard,
)

# Test against our partitioned testing data
accuracy_score = h.test(testing_data)
print(f"Accuracy: {accuracy_score * 100:.2f}%")
```

In [ ]:
# TODO 8.5 — Configure and run a Hyperparameter object

# TODO 8.5-a: instantiate Hyperparameter with k=5, distance=manhattan_distance,
#             training=training_data, algorithm=k_nn_standard
h = ...

# TODO 8.5-b: call .test(testing_data) to compute accuracy
accuracy_score = ...

print(f"Accuracy: {accuracy_score * 100:.2f}%")

In [ ]:
# Quick check — accuracy must be in [0, 1] and (for this dataset) reasonably high.
assert 0.0 <= accuracy_score <= 1.0
print(f"OK: accuracy = {accuracy_score:.4f}  (in [0, 1])")
# With dedup-aware bucketing on iris-like data, accuracy should be ≥ 0.7.
assert accuracy_score >= 0.7, f"accuracy too low: {accuracy_score}"
print(f"OK: accuracy ≥ 0.7 on the 1-in-5 bucket split")
print("Section 8.5 passed.")

### 8.6 — Bridge: `timeit` Comparison (Slide 41) + Equivalence Assertion

Two questions to answer:

1. **Do both algorithms agree?** — `Hyperparameter.test()` should return the SAME accuracy whether we plug in `k_nn_standard` or `k_nn_bisect`. We assert this directly.
2. **Which is faster?** — `timeit` measures wall-clock time. With only 24 training samples the difference is small; on bigger streams `k_nn_bisect` pulls ahead because it never sorts the full distance array.

### Full code reference (Slide 41)

```python
import timeit

setup_code = "from __main__ import h_std, h_bisect, testing_data"
time_std = timeit.timeit(
    "h_std.test(testing_data)",
    setup=setup_code, number=10,
)
time_bisect = timeit.timeit(
    "h_bisect.test(testing_data)",
    setup=setup_code, number=10,
)
print(f"Standard: {time_std:.2f}s | Bisect: {time_bisect:.2f}s")
```

In [ ]:
# TODO 8.6 — Profile standard vs bisect with timeit
import timeit

# Both Hyperparameter configurations share k, distance, and training set; only `algorithm` differs.
h_std    = Hyperparameter(k=5, distance=manhattan_distance,
                           training=training_data, algorithm=k_nn_standard)
h_bisect = Hyperparameter(k=5, distance=manhattan_distance,
                           training=training_data, algorithm=k_nn_bisect)

# TODO 8.6-a: time h_std.test(testing_data) with number=10 (use globals=globals() so the
#             names h_std, h_bisect, testing_data resolve from this notebook).
time_std = timeit.timeit(
    "h_std.test(testing_data)",
    globals=globals(), number=...,
)

# TODO 8.6-b: time h_bisect.test(testing_data) the same way
time_bisect = ...

print(f"Standard: {time_std:.4f}s | Bisect: {time_bisect:.4f}s")

In [ ]:
# Quick check — both algorithms must agree on accuracy (correctness > performance).
acc_std    = h_std.test(testing_data)
acc_bisect = h_bisect.test(testing_data)
assert acc_std == acc_bisect, \
    f"the two algorithms disagree on accuracy: std={acc_std}, bisect={acc_bisect}"
print(f"OK: both algorithms produced identical accuracy = {acc_std:.4f}")

# Sanity-check that timeit actually measured something positive.
assert time_std    > 0, "timeit returned 0 — measurement failed"
assert time_bisect > 0, "timeit returned 0 — measurement failed"
print(f"OK: timeit measured Standard={time_std:.4f}s, Bisect={time_bisect:.4f}s over 10 runs")
print("Section 8.6 passed.")

## 9. Summary (Slide 44)

> *"The Iterator Pattern abstracts traversal logic from container structures. Supported natively by Python's `for`, `in`, and mapping tools. Accelerated via Comprehensions (List, Dict, Set). Optimized for massive data via Generators (`yield`) and Pipelines."*

### What you built

| Slide   | Topic                                        | What you wrote                                                       |
| ------- | -------------------------------------------- | -------------------------------------------------------------------- |
| 6, 8    | Iterator protocol + manual loop              | `CapitalIterable`, `CapitalIterator`, hand-driven `iter()`/`next()`  |
| 11, 13  | List comprehensions (basic + filtered)       | `[int(num) for num in ...]`, `even_squares`, TSV → dicts             |
| 15      | Set / dict comprehensions                    | unique authors, `title → Book` map                                   |
| 16, 17  | Generator expression — lazy file filter      | memory comparison observation, `(l for l in infile if ...)`          |
| 19, 21  | `yield` generator function                   | `warnings_filter`, file-to-file pipeline                             |
| 23      | `yield from` delegation                      | refactored single-expression warnings filter                         |
| 25, 27  | Generator pipelines                          | 4-stage `lines → warn → split → tuple → csv`                         |
| 31, 33  | K-NN bucketing + chain split                 | `partition_data`, `split_partitions`                                 |
| 37, 39  | K-NN brute force + bisect optimization       | `k_nn_standard`, `k_nn_bisect`                                       |
| 35, 41  | `Hyperparameter.test()` + `timeit` profile   | accuracy comparison + wall-clock benchmark                           |

### Cleanup

In [ ]:
# Tidy up every file we wrote.
from pathlib import Path
for name in (
    "data.tsv",
    "massive_server_log.txt", "warnings_only.log",
    "input.txt", "output.txt",
    "log.txt", "output.csv",
):
    Path(name).unlink(missing_ok=True)
print("Cleaned up all temp files.")

Great work! 🎯 You now have the full K-NN data path end-to-end: Lecture 5's split → Lecture 6.1's higher-order partition → Lecture 6.2's ingestion readers → **Lecture 7.1's classification** with two competing algorithms benchmarked side by side.